In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [11]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType

In [52]:
import csv
import json
import os
import sys
import time
from datetime import date, timedelta
from loguru import logger
from dotenv import load_dotenv
from google.cloud import storage
from utils.api_utils import WeatherAPI
from config import GCS_BUCKET
import sys, os
from pyspark.sql.functions import col, from_json, to_timestamp,lit,concat,now,date_format,current_timestamp,date_trunc
load_dotenv()



True

In [3]:
from spark_session import get_spark
spark = get_spark("BronzeProcessing")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/TrueTruwowng/.ivy2/cache
The jars for the packages stored in: /home/TrueTruwowng/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-63d8795a-2d65-4924-9f47-28a1a4d02baf;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 932ms :: artifacts dl 40ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |

26/05/15 06:19:46 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [13]:
weather_landing_dir = f"gs://{GCS_BUCKET}/landing/weather"

In [21]:
weather_schema = StructType([
    StructField("road_id",        StringType(),  True),
    StructField("road_name",      StringType(),  True),
    StructField("datetime",           StringType(),  True),
    StructField("temperature",    DoubleType(),  True),
    StructField("windspeed",      DoubleType(),  True),
    StructField("humidity",       IntegerType(), True),
    StructField("precipitation",  DoubleType(),  True),
    StructField("weathercode",    IntegerType(), True),
    StructField("weather",        StringType(),  True),
    StructField("ingestion_time", StringType(),  True),
])
weather_df = spark.read.json(f"{weather_landing_dir}/2026-01-05.json", schema=weather_schema)

In [24]:
weather_df = weather_df.select(col("datetime"), col("road_id"), col("road_name"), col("temperature"), col("weather"), col("windspeed"),col("precipitation"))
weather_df.show()

+----------------+---------+----------------+-----------+-------------+---------+-------------+
|        datetime|  road_id|       road_name|temperature|      weather|windspeed|precipitation|
+----------------+---------+----------------+-----------+-------------+---------+-------------+
|2026-01-05T00:00|163053434|395 Nguyễn Khang|       17.0|     Overcast|      7.2|          0.0|
|2026-01-05T01:00|163053434|395 Nguyễn Khang|       17.0|     Overcast|      7.6|          0.0|
|2026-01-05T02:00|163053434|395 Nguyễn Khang|       17.1|Light drizzle|      8.1|          0.2|
|2026-01-05T03:00|163053434|395 Nguyễn Khang|       17.0|Light drizzle|      7.6|          0.1|
|2026-01-05T04:00|163053434|395 Nguyễn Khang|       16.8|Light drizzle|      8.5|          0.2|
|2026-01-05T05:00|163053434|395 Nguyễn Khang|       16.7|Light drizzle|     10.9|          0.2|
|2026-01-05T06:00|163053434|395 Nguyễn Khang|       16.3|Light drizzle|     15.0|          0.3|
|2026-01-05T07:00|163053434|395 Nguyễn K

In [42]:
traffic_schema = StructType([
    StructField("id",              StringType(),    True),
    StructField("road_name",       StringType(),    True),
    StructField("recordDatetime",  TimestampType(), True),
    StructField("laneDensity",     DoubleType(),    True),
    StructField("occupancy",       DoubleType(),    True),
    StructField("waitingTime",     DoubleType(),    True),
    StructField("speed",           DoubleType(),    True),
    StructField("sampledSeconds",  DoubleType(),    True),
    StructField("traveltime",      DoubleType(),    True),
    StructField("flow",            DoubleType(),    True),
    StructField("processDatetime", StringType(),    True),
])

In [61]:
traffic_df = spark.read.json(f"gs://{GCS_BUCKET}/landing/traffic/2026-05-04/2026-05-04_0003.json")
traffic_df.show()


+-------+-----+----------+-------+--------+-----+-------+-----+-------------+---------------+-------------+-----------+----+---------+--------------+-----------------+--------------------+--------------+-----+-------------+--------+----------+-----------+
|arrived|begin|      date|density|departed|  end|entered| flow|           id|laneChangedFrom|laneChangedTo|laneDensity|left|occupancy|overlapDensity|overlapTraveltime|           road_name|sampledSeconds|speed|speedRelative|timeLoss|traveltime|waitingTime|
+-------+-----+----------+-------+--------+-----+-------+-----+-------------+---------------+-------------+-----------+----+---------+--------------+-----------------+--------------------+--------------+-----+-------------+--------+----------+-----------+
|      0|00:03|2026-05-04|   1.89|       0|00:06|      1|20.00|-1008268818#0|              0|            0|       1.89|   1|     0.98|          2.03|            21.75|   Phố Phạm Tuấn Tài|         21.75| 2.97|         0.21|   16.85|

In [63]:
traffic_df = traffic_df.select(
    col("id"),
    col("road_name"),
    date_format(
    concat(col("date"), lit("T"), col("end")).cast("timestamp"),
    "yyyy-MM-dd'T'HH:mm"
).alias("recordDatetime"),
    col("density").cast(DoubleType()),
    col("occupancy").cast(DoubleType()),
    col("waitingTime").cast(DoubleType()),
    col("speed").cast(DoubleType()),
    col("sampledSeconds").cast(DoubleType()),
    col("traveltime").cast(DoubleType()),
    col("flow").cast(DoubleType()),
    col("entered").cast(DoubleType()),
    col("left").cast(DoubleType()),
    col("timeloss"),
    date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm").alias("processDatetime"),
)

In [64]:
traffic_df.show()

+-------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+
|           id|           road_name|  recordDatetime|density|occupancy|waitingTime|speed|sampledSeconds|traveltime|flow|entered|left|timeloss| processDatetime|
+-------------+--------------------+----------------+-------+---------+-----------+-----+--------------+----------+----+-------+----+--------+----------------+
|-1008268818#0|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   1.89|     0.98|       13.0| 2.97|         21.75|     20.32|20.0|    1.0| 1.0|   16.85|2026-05-15T14:11|
|-1008268818#1|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   0.43|     0.22|        0.0|12.92|          0.68|       0.3|20.0|    1.0| 1.0|    0.04|2026-05-15T14:11|
|-1008268818#4|   Phố Phạm Tuấn Tài|2026-05-04T00:06|   0.41|     0.21|        0.0|13.53|          0.87|       0.5|20.0|    1.0| 1.0|    0.02|2026-05-15T14:11|
|-1119291005#1|Ngõ 6 Phố Trần Qu...|2026

In [65]:
traffic_df.write.mode("overwrite")    .option("overwriteSchema", "true") \
.save(f"gs://{GCS_BUCKET}/bronze/traffic/2026-05-04", format="delta")

26/05/15 07:13:31 WARN TaskSetManager: Lost task 0.0 in stage 52.0 (TID 245) (10.148.0.3 executor 1): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to gs://big-data-storage13/bronze/traffic/2026-05-04.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:774)
	at org.apache.spark.sql.delta.files.DeltaFileFormatWriter$.executeTask(DeltaFileFormatWriter.scala:447)
	at org.apache.spark.sql.delta.files.DeltaFileFormatWriter$.$anonfun$executeWrite$2(DeltaFileFormatWriter.scala:274)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:93)
	at org.apache.spark.TaskContext.runTaskWithListeners(TaskContext.scala:166)
	at org.apache.spark.scheduler.Task.run(Task.scala:141)
	at org.apache.spark.executor.Executor$TaskRunner.$anonfun$run$4(Executor.scala:620)
	at org.apache.spark.util.SparkErrorUtils.tryWithSafeFinally(SparkErrorUtils.scala:64)
	at org.apache.spark.util.SparkErrorUtils.tr